# 第八章：记忆与检索

本章介绍 HelloAgents 框架中的两大核心工具：**MemoryTool（记忆工具）** 和 **RAGTool（检索增强生成工具）**。

---

## 本章结构

| 节次 | 内容 |
|:----:|------|
| 1 | 环境准备与工具安装 |
| 2 | MemoryTool：四种记忆类型 |
| 3 | MemoryTool：基础操作（增删改查） |
| 4 | MemoryTool：记忆整合机制 |
| 5 | RAGTool：文档处理管道 |
| 6 | RAGTool：智能问答 |
| 7 | Agent 集成 MemoryTool + RAGTool |

---

## 核心概念速览

```
人类记忆系统               HelloAgents 对应实现
──────────────────────────────────────────────
工作记忆（短暂、有限）    →  WorkingMemory  （TTL + 容量限制）
情景记忆（事件、经历）    →  EpisodicMemory （时间戳 + 事件标签）
语义记忆（知识、概念）    →  SemanticMemory （概念 + 领域分类）
感知记忆（文件、图像）    →  PerceptualMemory（文件哈希去重）
──────────────────────────────────────────────
外部知识库检索            →  RAGTool（Any→Markdown→分块→向量化）
```

## 1. 环境准备

In [1]:
# 安装依赖
import subprocess, sys

packages = ["hello-agents==0.1.1", "python-dotenv", "markitdown"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ 依赖安装完成")

✅ 依赖安装完成


In [2]:
import os
from dotenv import load_dotenv


load_dotenv()
key = os.getenv("MODELSCOPE_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("LLM_API_KEY")
if key:
    print(f"✅ API 密钥已加载: {'*' * 8 + key[-4:]}")
else:
    print("⚠️  未检测到 API 密钥，RAGTool 问答功能无法运行")

✅ API 密钥已加载: ********XJYQ


## 2. MemoryTool：四种记忆类型

HelloAgents 模仿人类认知心理学，设计了四种记忆类型：

| 记忆类型 | 对应人类记忆 | 特点 | 适用场景 |
|----------|-------------|------|----------|
| `working` | 工作记忆 | 容量有限(50条)、TTL自动过期 | 当前对话上下文 |
| `episodic` | 情景记忆 | 带时间戳、事件标签 | 历史对话记录 |
| `semantic` | 语义记忆 | 概念+领域分类 | 长期知识存储 |
| `perceptual` | 感知记忆 | 文件哈希去重 | 处理过的文件记录 |

### 架构图
```
MemoryTool
  ├── WorkingMemory    → 纯内存存储，速度最快，有容量上限
  ├── EpisodicMemory   → 带时间轴的事件记录
  ├── SemanticMemory   → 持久化知识图谱
  └── PerceptualMemory → 基于文件哈希的去重存储
```

In [ ]:
import time, uuid, re, math
from collections import defaultdict
from datetime import datetime

class MemoryTool:
    """四种记忆类型的统一管理工具，模拟人类认知记忆系统（纯 Python 实现）。"""
    WORKING_CAPACITY = 50

    def __init__(self, user_id: str, memory_types: list = None):
        self.user_id      = user_id
        self.memory_types = memory_types or ["working", "episodic", "semantic", "perceptual"]
        self._stores: dict = {t: [] for t in self.memory_types}

    def run(self, params: dict) -> str:
        action = params.get("action", "")
        return {
            "add":         self._add,
            "search":      self._search,
            "summary":     self._summary,
            "stats":       self._stats,
            "update":      self._update,
            "remove":      self._remove,
            "forget":      self._forget,
            "consolidate": self._consolidate,
            "clear_all":   self._clear_all,
        }.get(action, lambda p: f"❌ 未知操作: {action}")(params)
    #get方法根据 action 调用对应的函数，如果 action 不在预定义的操作列表中，则返回一个默认的 lambda 函数，输出错误信息。
    #如果action是 "add"，则调用 self._add(params) 来处理添加记忆的逻辑；如果是 "search"，则调用 self._search(params) 来处理搜索记忆的逻辑，以此类推。

    def _add(self, p):
        mtype = p.get("memory_type", "working")
        if mtype not in self._stores:
            return f"❌ 记忆类型 '{mtype}' 未初始化"
        record = {
            "id": str(uuid.uuid4())[:8], # 唯一标识符（前8位）
            "content": p.get("content", ""), # 记忆内容
            "memory_type": mtype,
            "importance": float(p.get("importance", 0.5)), # 重要性评分
            "created_at": time.time(), # 精确时间戳销
            "timestamp": datetime.now().isoformat(timespec="seconds") # 可读格式时间
        }
        # 将其他参数也存入记录，方便后续检索和统计
        for k, v in p.items():
            if k not in ("action", "content", "memory_type", "importance"):
                record[k] = v
        store = self._stores[mtype] #取出对应类型的记忆存储列表
        # 工作记忆有容量限制，重要性最低的记忆会被遗忘
        if mtype == "working" and len(store) >= self.WORKING_CAPACITY:
            store.sort(key=lambda x: x["importance"])
            store.pop(0) #删除第一条（重要性最低的）记忆
        store.append(record)
        return f"✅ 已存入{mtype}记忆 [id={record['id']}，重要性={record['importance']}]"

    def _tfidf_score(self, query, text):
        """
        计算 query 与 text 之间的简化相关度评分。

        原理：统计 text 中有多少词出现在 query 词集合里，
        再除以 text 的总词数，得到一个 [0, 1] 范围的命中率。

        score = (text 中命中 query 的词数) / (text 总词数)

        注意：这是对 TF-IDF 的简化实现，并非严格意义上的 TF-IDF，
        侧重衡量 text 对 query 词汇的"覆盖密度"。
        """
        # 将 query 分词并转为集合，便于 O(1) 查找，同时自动去重
        q_words = set(re.findall(r'\w+', query.lower()))

        # 将 text 分词为列表，保留重复词（用于计算词频分母）
        t_words = re.findall(r'\w+', text.lower())

        # 若 text 为空，无法计算分数，返回 0
        if not t_words:
            return 0.0

        # 统计 text 中属于 query 词集合的词数，除以 text 总词数
        # 结果表示：text 中有多少比例的词与查询相关
        return sum(1 for w in t_words if w in q_words) / len(t_words)

    def _search(self, p):
        """
        在记忆库中检索与 query 最相关的记忆条目。

        评分策略：相关度 = TF-IDF 命中率 + 重要性 × 0.2
        - TF-IDF 命中率反映内容与查询的文本相关性
        - 重要性权重 0.2 确保高重要性记忆在相关度相近时优先排序
        结果按综合评分从高到低排序，返回前 limit 条。
        """
        query  = p.get("query", "")
        mtype  = p.get("memory_type")           # 可选：指定记忆类型，不指定则搜全部
        limit  = int(p.get("limit", 5))          # 返回结果条数上限，默认 5

        # 若指定了记忆类型，只在该类型中搜索；否则搜索全部类型
        stores = [self._stores[mtype]] if mtype and mtype in self._stores \
                 else list(self._stores.values())

        candidates = []
        for store in stores:
            for rec in store:
                # 综合评分 = 文本相关度 + 重要性加权
                score = self._tfidf_score(query, rec["content"]) + rec["importance"] * 0.2
                candidates.append((score, rec))

        # 按评分从高到低排序，截取前 limit 条
        top = sorted(candidates, key=lambda x: -x[0])[:limit]
        if not top:
            return "未找到相关记忆"

        return "\n".join(
            f"[{r['memory_type']}] {r['content'][:80]}（重要性={r['importance']}, 相关度={s:.3f}）"
            for s, r in top
        )

    def _summary(self, p):
        """
        生成用户记忆的可读摘要。

        按记忆类型分组，每组展示最近的 limit 条记忆（按创建时间倒序），
        方便快速了解各类记忆的整体内容。
        """
        limit = int(p.get("limit", 10))  # 每种类型展示的最近记忆条数，默认 10
        lines = [f"📋 用户 {self.user_id} 的记忆摘要："]
        for mtype, store in self._stores.items():
            # 按创建时间倒序，取最近 limit 条
            recent = sorted(store, key=lambda x: -x["created_at"])[:limit]
            lines.append(f"\n  [{mtype}] 共 {len(store)} 条，最近 {len(recent)} 条：")
            for r in recent:
                lines.append(f"    · {r['content'][:60]}")
        return "\n".join(lines)

    def _stats(self, p):
        """
        统计各记忆类型的数量和平均重要性。

        返回格式化的统计报告，包含：
        - 总记忆条数
        - 每种类型的条数
        - 每种类型的平均重要性（存储为空时显示 0.00）
        """
        total = sum(len(s) for s in self._stores.values())  # 所有类型记忆总数
        lines = [f"📊 记忆统计 (user={self.user_id})  总计: {total} 条"]
        for mtype, store in self._stores.items():
            # 计算平均重要性；存储为空时避免除以零
            avg = (sum(r["importance"] for r in store) / len(store)) if store else 0
            lines.append(f"  {mtype:12s}: {len(store):3d} 条  平均重要性={avg:.2f}")
        return "\n".join(lines)

    def _update(self, p):
        """
        根据 id 更新指定记忆的字段。

        遍历所有记忆类型，找到匹配 id 的记录后，
        用传入参数覆盖对应字段（action 和 id 字段不参与覆盖）。
        若找不到对应 id，返回错误提示。
        """
        rid = p.get("id")  # 目标记忆的唯一标识符
        for store in self._stores.values():
            for rec in store:
                if rec["id"] == rid:
                    # 排除 action 和 id 字段，其余字段全量更新
                    rec.update({k: v for k, v in p.items() if k not in ("action", "id")})
                    return f"✅ 记忆 [id={rid}] 已更新"
        return f"❌ 未找到 id={rid}"

    def _remove(self, p):
        """
        根据 id 从记忆库中永久删除指定记忆。

        遍历所有记忆类型，找到匹配记录后立即移除并返回成功信息。
        与 forget 不同，remove 是精确删除单条，forget 是按阈值批量清除。
        """
        rid = p.get("id")  # 目标记忆的唯一标识符
        for key, store in self._stores.items():
            for rec in store:
                if rec["id"] == rid:
                    store.remove(rec)  # 从列表中移除该记录
                    return f"✅ 已从 {key} 删除 [id={rid}]"
        return f"❌ 未找到 id={rid}"

    def _forget(self, p):
        """
        批量遗忘重要性低于阈值的记忆，模拟人类遗忘低价值信息的机制。

        对所有记忆类型执行过滤，保留重要性 >= importance_threshold 的记录，
        丢弃其余记录。默认阈值为 0.4。
        """
        thr = float(p.get("importance_threshold", 0.4))  # 重要性阈值，默认 0.4
        removed = 0
        for key, store in self._stores.items():
            before = len(store)
            # 仅保留重要性达到阈值的记忆
            self._stores[key] = [r for r in store if r["importance"] >= thr]
            removed += before - len(self._stores[key])  # 统计本类型删除数量
        return f"✅ 已遗忘 {removed} 条重要性 < {thr} 的记忆"

    def _consolidate(self, p):
        """
        将短期记忆中的重要内容整合（升级）到长期记忆，模拟人类睡眠巩固记忆的过程。

        规则：
        - 仅整合重要性 >= importance_threshold 的记忆（默认 0.7）
        - 整合后重要性提升 ×1.1（上限 1.0），体现巩固加权
        - 原记忆保留，在目标类型中新建一条副本，并标记 consolidated_from 来源
        """
        src = p.get("source_type", "working")          # 来源记忆类型，默认工作记忆
        tgt = p.get("target_type", "episodic")          # 目标记忆类型，默认情景记忆
        thr = float(p.get("importance_threshold", 0.7)) # 整合的重要性阈值，默认 0.7

        if src not in self._stores or tgt not in self._stores:
            return "❌ 记忆类型不存在"

        count = 0
        for rec in self._stores[src]:
            if rec["importance"] >= thr:
                # 构造新记录：复制原记录，更新 id / 类型 / 重要性 / 来源标记
                new = {
                    **rec,
                    "id": str(uuid.uuid4())[:8],        # 新的唯一 id
                    "memory_type": tgt,
                    "importance": min(1.0, rec["importance"] * 1.1),  # 重要性提升，不超过 1.0
                    "consolidated_from": src            # 记录整合来源，便于追溯
                }
                self._stores[tgt].append(new)
                count += 1
        return f"✅ 已将 {count} 条 {src} 记忆（重要性>={thr}）整合到 {tgt}"

    def _clear_all(self, p):
        """
        清空所有记忆类型的全部记录，不可恢复。
        通常用于重置会话状态或测试场景。
        """
        for key in self._stores:
            self._stores[key] = []  # 将每种记忆类型的存储列表置空
        return "✅ 所有记忆已清空"


# ── 初始化验证 ──
memory_tool = MemoryTool(
    user_id="tutorial_user",
    memory_types=["working", "episodic", "semantic", "perceptual"]
)
print("✅ MemoryTool 初始化完成")
print("📋 可用操作: add, search, summary, stats, update, remove, forget, consolidate, clear_all")
print()
print(memory_tool.run({"action": "stats"}))


✅ MemoryTool 初始化完成
📋 可用操作: add, search, summary, stats, update, remove, forget, consolidate, clear_all

📊 记忆统计 (user=tutorial_user)  总计: 0 条
  working     :   0 条  平均重要性=0.00
  episodic    :   0 条  平均重要性=0.00
  semantic    :   0 条  平均重要性=0.00
  perceptual  :   0 条  平均重要性=0.00


### MemoryTool 数据结构速览

`MemoryTool` 的核心是一个字典 `_stores`，键是记忆类型名，值是**记忆条目列表**：

```
_stores = {
    "working":    [ record, record, ... ],   # 最多 50 条，超出自动淘汰
    "episodic":   [ record, record, ... ],
    "semantic":   [ record, record, ... ],
    "perceptual": [ record, record, ... ],
}
```

每条 **record（记忆条目）** 是一个普通字典，固定字段如下：

| 字段 | 类型 | 含义 |
|------|------|------|
| `id` | str | UUID 前 8 位，唯一标识 |
| `content` | str | 记忆的文本内容 |
| `memory_type` | str | 记忆类型（working / episodic / semantic / perceptual） |
| `importance` | float | 重要性评分，范围 0.0 ~ 1.0 |
| `created_at` | float | Unix 时间戳（精确，用于排序） |
| `timestamp` | str | 可读时间字符串（如 `2024-01-01T12:00:00`） |
| *(任意扩展字段)* | any | 用户传入的额外元数据，如 `event_type`、`domain` 等 |

> 运行下方单元格，直接打印真实的数据结构。


In [ ]:
import json

# 构造一个演示用的 MemoryTool 并写入几条记忆
demo = MemoryTool(user_id="demo", memory_types=["working", "episodic", "semantic"])

demo.run({"action": "add", "content": "正在处理用户的问题", "memory_type": "working",
          "importance": 0.6, "task_type": "dialog"})
demo.run({"action": "add", "content": "用户昨天询问过 Python 语法", "memory_type": "episodic",
          "importance": 0.8, "event_type": "conversation", "location": "online"})
demo.run({"action": "add", "content": "Python 是解释型语言", "memory_type": "semantic",
          "importance": 0.9, "concept": "python_basics", "domain": "programming"})

# ── 打印完整 _stores 结构 ──
print("=" * 60)
print("_stores 完整结构（每个 key 对应一个记忆列表）:")
print("=" * 60)
for mtype, records in demo._stores.items():
    print(f"\n📂 _stores['{mtype}']  →  共 {len(records)} 条")
    for rec in records:
        print(json.dumps(rec, ensure_ascii=False, indent=4))

# ── 单条记录字段说明 ──
print("\n" + "=" * 60)
print("单条 record 的字段结构（以 episodic 为例）:")
print("=" * 60)
rec = demo._stores["episodic"][0]
for k, v in rec.items():
    note = {
        "id":           "← UUID 前8位，唯一标识",
        "content":      "← 记忆文本内容",
        "memory_type":  "← 所属记忆类型",
        "importance":   "← 重要性评分 [0,1]",
        "created_at":   "← Unix 时间戳（浮点，用于排序）",
        "timestamp":    "← 可读格式时间",
        "event_type":   "← 扩展字段（用户自定义）",
        "location":     "← 扩展字段（用户自定义）",
    }.get(k, "← 扩展字段（用户自定义）")
    print(f"  {k:15s}: {repr(v):35s}  {note}")


## 3. MemoryTool：基础操作

MemoryTool 统一通过 `.run(dict)` 接口操作，`action` 字段决定执行什么动作：

```python
memory_tool.run({
    "action": "add",         # 操作类型
    "content": "记忆内容",
    "memory_type": "working", # 存入哪种记忆
    "importance": 0.8,        # 重要性 0~1
    # ... 其他元数据字段（任意扩展）
})
```

In [5]:
# MemoryTool 类已在上方 cell 定义，直接使用即可

memory_tool = MemoryTool(
    user_id="tutorial_user_ops",
    memory_types=["working", "episodic", "semantic", "perceptual"]
)

# === 添加四种记忆 ===

# 1. 工作记忆：当前任务上下文
r1 = memory_tool.run({
    "action": "add",
    "content": "正在学习 HelloAgents 框架的记忆系统",
    "memory_type": "working",
    "importance": 0.7,
    "task_type": "learning"
})
print(f"工作记忆: {r1}")

# 2. 情景记忆：历史事件
r2 = memory_tool.run({
    "action": "add",
    "content": "2024年开始深入研究 AI Agent 技术",
    "memory_type": "episodic",
    "importance": 0.8,
    "event_type": "milestone",
    "location": "研发中心"
})
print(f"情景记忆: {r2}")

# 3. 语义记忆：知识概念
r3 = memory_tool.run({
    "action": "add",
    "content": "记忆系统包括工作记忆、情景记忆、语义记忆和感知记忆四种类型",
    "memory_type": "semantic",
    "importance": 0.9,
    "concept": "memory_types",
    "domain": "cognitive_science"
})
print(f"语义记忆: {r3}")

# 4. 感知记忆：文件/文档记录
r4 = memory_tool.run({
    "action": "add",
    "content": "查看了记忆系统的架构图和实现代码",
    "memory_type": "perceptual",
    "importance": 0.6,
    "modality": "document",
    "source": "technical_documentation"
})
print(f"感知记忆: {r4}")


工作记忆: ✅ 已存入working记忆 [id=42860ef9，重要性=0.7]
情景记忆: ✅ 已存入episodic记忆 [id=8b5e5b99，重要性=0.8]
语义记忆: ✅ 已存入semantic记忆 [id=baad9ba0，重要性=0.9]
感知记忆: ✅ 已存入perceptual记忆 [id=f83fc39a，重要性=0.6]


In [6]:
# === 搜索记忆 ===
# 底层使用 TF-IDF 语义检索 + 关键词匹配 + 时间衰减 + 重要性权重的混合策略

print("🔍 搜索演示")
print("-" * 40)

# 跨类型搜索（默认搜全部）
result = memory_tool.run({
    "action": "search",
    "query": "记忆系统",
    "limit": 5
})
print(f"跨类型搜索 '记忆系统':\n{result}")

print()

# 指定记忆类型搜索
result2 = memory_tool.run({
    "action": "search",
    "query": "AI Agent",
    "memory_type": "episodic",
    "limit": 3
})
print(f"在情景记忆中搜索 'AI Agent':\n{result2}")

🔍 搜索演示
----------------------------------------
跨类型搜索 '记忆系统':
[semantic] 记忆系统包括工作记忆、情景记忆、语义记忆和感知记忆四种类型（重要性=0.9, 相关度=0.180）
[episodic] 2024年开始深入研究 AI Agent 技术（重要性=0.8, 相关度=0.160）
[working] 正在学习 HelloAgents 框架的记忆系统（重要性=0.7, 相关度=0.140）
[perceptual] 查看了记忆系统的架构图和实现代码（重要性=0.6, 相关度=0.120）

在情景记忆中搜索 'AI Agent':
[episodic] 2024年开始深入研究 AI Agent 技术（重要性=0.8, 相关度=0.660）


In [7]:
# === 摘要与统计 ===

print("📋 记忆摘要")
summary = memory_tool.run({"action": "summary", "limit": 10})
print(summary)

print("\n📊 记忆统计")
stats = memory_tool.run({"action": "stats"})
print(stats)

📋 记忆摘要
📋 用户 tutorial_user_ops 的记忆摘要：

  [working] 共 1 条，最近 1 条：
    · 正在学习 HelloAgents 框架的记忆系统

  [episodic] 共 1 条，最近 1 条：
    · 2024年开始深入研究 AI Agent 技术

  [semantic] 共 1 条，最近 1 条：
    · 记忆系统包括工作记忆、情景记忆、语义记忆和感知记忆四种类型

  [perceptual] 共 1 条，最近 1 条：
    · 查看了记忆系统的架构图和实现代码

📊 记忆统计
📊 记忆统计 (user=tutorial_user_ops)  总计: 4 条
  working     :   1 条  平均重要性=0.70
  episodic    :   1 条  平均重要性=0.80
  semantic    :   1 条  平均重要性=0.90
  perceptual  :   1 条  平均重要性=0.60


In [8]:
# === 遗忘机制 ===
# forget 会按重要性阈值清除低价值记忆，模拟人类「遗忘」低优先级内容

forget_demo = MemoryTool(
    user_id="forget_demo_user",
    memory_types=["working", "episodic"]
)

# 添加不同重要性的记忆
items = [
    ("高优先级：深度学习核心原理", 0.9),
    ("中优先级：今天开了个会",     0.5),
    ("低优先级：刷了5分钟微博",    0.2),
    ("低优先级：喝了杯水",         0.1),
]
for content, importance in items:
    forget_demo.run({
        "action": "add",
        "content": content,
        "memory_type": "working",
        "importance": importance
    })

print("遗忘前：")
print(forget_demo.run({"action": "stats"}))

# forget：清除重要性 < 0.4 的记忆
result = forget_demo.run({
    "action": "forget",
    "importance_threshold": 0.4
})
print(f"\n遗忘操作结果: {result}")
print("\n遗忘后：")
print(forget_demo.run({"action": "stats"}))


遗忘前：
📊 记忆统计 (user=forget_demo_user)  总计: 4 条
  working     :   4 条  平均重要性=0.42
  episodic    :   0 条  平均重要性=0.00

遗忘操作结果: ✅ 已遗忘 2 条重要性 < 0.4 的记忆

遗忘后：
📊 记忆统计 (user=forget_demo_user)  总计: 2 条
  working     :   2 条  平均重要性=0.70
  episodic    :   0 条  平均重要性=0.00


## 4. 记忆整合机制

**整合（Consolidation）** = 把短期的工作记忆「升级」为长期记忆的过程，类似人类睡眠期间大脑巩固白天经历。

```
工作记忆 ──(重要性 >= 阈值)──→ 情景记忆
             ↑
        consolidate 操作
```

整合规则：
- 重要性 >= `threshold`（默认 0.7）的工作记忆才会被整合
- 整合后重要性提升 ×1.1（加权奖励）
- 原工作记忆保留，新情景记忆被创建

In [9]:
consol_tool = MemoryTool(
    user_id="consolidation_demo",
    memory_types=["working", "episodic", "semantic", "perceptual"]
)

# 添加不同重要性的工作记忆
memories = [
    ("学习了 Transformer 架构原理",     0.9, "deep_learning"),
    ("完成了 Python 代码调试任务",       0.8, "programming"),
    ("参加了团队会议",                   0.7, "teamwork"),
    ("查看了今天天气预报",               0.3, "daily_life"),
    ("阅读了注意力机制论文",             0.85, "research"),
    ("喝了一杯咖啡",                     0.2, "daily_life"),
]

print("📝 添加工作记忆...")
for content, importance, topic in memories:
    consol_tool.run({
        "action": "add",
        "content": content,
        "memory_type": "working",
        "importance": importance,
        "topic": topic
    })
    print(f"  [重要性={importance}] {content}")

print("\n整合前统计:")
print(consol_tool.run({"action": "stats"}))

# 执行整合：重要性 >= 0.7 的工作记忆转为情景记忆
print("\n🔄 执行记忆整合（阈值=0.7）...")
result = consol_tool.run({
    "action": "consolidate",
    "source_type": "working",
    "target_type": "episodic",
    "importance_threshold": 0.7
})
print(f"整合结果: {result}")

print("\n整合后统计:")
print(consol_tool.run({"action": "stats"}))


📝 添加工作记忆...
  [重要性=0.9] 学习了 Transformer 架构原理
  [重要性=0.8] 完成了 Python 代码调试任务
  [重要性=0.7] 参加了团队会议
  [重要性=0.3] 查看了今天天气预报
  [重要性=0.85] 阅读了注意力机制论文
  [重要性=0.2] 喝了一杯咖啡

整合前统计:
📊 记忆统计 (user=consolidation_demo)  总计: 6 条
  working     :   6 条  平均重要性=0.62
  episodic    :   0 条  平均重要性=0.00
  semantic    :   0 条  平均重要性=0.00
  perceptual  :   0 条  平均重要性=0.00

🔄 执行记忆整合（阈值=0.7）...
整合结果: ✅ 已将 4 条 working 记忆（重要性>=0.7）整合到 episodic

整合后统计:
📊 记忆统计 (user=consolidation_demo)  总计: 10 条
  working     :   6 条  平均重要性=0.62
  episodic    :   4 条  平均重要性=0.89
  semantic    :   0 条  平均重要性=0.00
  perceptual  :   0 条  平均重要性=0.00


## 5. RAGTool：文档处理管道

RAG = **R**etrieval-**A**ugmented **G**eneration（检索增强生成）

**RAGTool 的处理管道：**
```
任意格式文件
  → MarkItDown 转换为 Markdown
     → 文本分块（Chunking）
        → TF-IDF 向量化
           → 本地向量数据库
              → 相似度检索
                 → LLM 生成答案
```

支持的文件格式：`.txt` `.md` `.pdf` `.docx` `.html` `.csv` `.json` 等（通过 MarkItDown 统一转换）

In [10]:
"""
RAGTool 纯 Python 实现
处理管道：文件读取 → Markdown 转换 → 文本分块 → TF-IDF 向量化 → 检索
（hello-agents 0.1.1 尚未内置，这里自行实现，API 与文档描述完全一致）
"""
import os, re, math, uuid, json
from pathlib import Path
from collections import defaultdict
from hello_agents import HelloAgentsLLM

class RAGTool:
    """检索增强生成工具：文档管道 → TF-IDF 检索 → LLM 问答。"""

    def __init__(self, knowledge_base_path: str = "./rag_kb", rag_namespace: str = "default"):
        self.kb_path   = Path(knowledge_base_path)
        self.namespace = rag_namespace
        self.kb_path.mkdir(parents=True, exist_ok=True)
        # 内存索引：chunks + TF-IDF
        self._chunks: list[dict] = []   # {id, text, metadata, source}
        self._vocab:  dict[str, int] = {}
        self._idf:    dict[str, float] = {}
        self._tfidf_matrix: list[dict[str, float]] = []  # per-chunk TF-IDF vectors
        self._load_index()

    # ─────────────────── 公共接口 ─────────────────────
    def run(self, params: dict) -> str:
        action = params.get("action", "")
        dispatch = {
            "add_document": self._add_document,
            "search":       self._search,
            "query":        self._query,
            "status":       self._status,
        }
        fn = dispatch.get(action)
        if not fn:
            return f"❌ 未知操作: {action}"
        return fn(params)

    # ─────────────────── add_document ─────────────────
    def _add_document(self, p: dict) -> str:
        file_path = p.get("file_path", "")
        metadata  = p.get("metadata", {})
        if not file_path or not Path(file_path).exists():
            return f"❌ 文件不存在: {file_path}"
        # 读取内容（统一按文本处理，支持 txt / md / json / csv）
        text = self._read_file(file_path)
        # 分块（按段落，最长 300 字）
        chunks = self._chunk_text(text)
        source = Path(file_path).name
        for chunk in chunks:
            self._chunks.append({
                "id":       str(uuid.uuid4())[:8],
                "text":     chunk,
                "metadata": metadata,
                "source":   source,
            })
        # 重新构建索引
        self._build_index()
        self._save_index()
        return f"✅ 已添加 {len(chunks)} 个分块 (来源: {source}，总块数: {len(self._chunks)})"

    # ─────────────────── search ───────────────────────
    def _search(self, p: dict) -> str:
        query = p.get("query", "")
        top_k = int(p.get("top_k", 3))
        if not self._chunks:
            return "知识库为空，请先添加文档"
        scores = self._cosine_scores(query)
        top = sorted(scores, key=lambda x: -x[1])[:top_k]
        if not top:
            return "未找到相关内容"
        lines = []
        for rank, (idx, score) in enumerate(top, 1):
            chunk = self._chunks[idx]
            lines.append(
                f"[{rank}] 来源: {chunk['source']}  相关度: {score:.3f}\n"
                f"    {chunk['text'][:200]}"
            )
        return "\n\n".join(lines)

    # ─────────────────── query (RAG) ──────────────────
    def _query(self, p: dict) -> str:
        query = p.get("query", "")
        top_k = int(p.get("top_k", 3))
        # 1. 检索
        context_str = self._search({"query": query, "top_k": top_k})
        # 2. LLM 生成
        try:
            llm = HelloAgentsLLM()
            prompt = (
                f"请根据以下参考资料回答问题，如资料中没有相关信息请如实说明。\n\n"
                f"参考资料：\n{context_str}\n\n问题：{query}"
            )
            answer = llm.invoke([{"role": "user", "content": prompt}])
            return answer
        except Exception as e:
            return f"[LLM 不可用，以下为检索结果]\n{context_str}\n\n(LLM 错误: {e})"

    # ─────────────────── status ───────────────────────
    def _status(self, p: dict) -> str:
        sources = defaultdict(int)
        for c in self._chunks:
            sources[c["source"]] += 1
        lines = [
            f"📊 RAGTool 状态 (namespace={self.namespace})",
            f"  总块数: {len(self._chunks)}",
            f"  词汇量: {len(self._vocab)}",
            f"  知识库路径: {self.kb_path}",
            "  已索引文件：",
        ]
        for src, cnt in sources.items():
            lines.append(f"    · {src}  ({cnt} 块)")
        return "\n".join(lines)

    # ─────────────────── 内部工具 ─────────────────────
    def _read_file(self, path: str) -> str:
        with open(path, encoding="utf-8", errors="ignore") as f:
            return f.read()

    def _chunk_text(self, text: str, max_len: int = 300) -> list[str]:
        """按段落分块，超长段落再切分。"""
        paragraphs = re.split(r'\n{2,}', text.strip())
        chunks = []
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            while len(para) > max_len:
                chunks.append(para[:max_len])
                para = para[max_len:]
            if para:
                chunks.append(para)
        return chunks or [text[:max_len]]

    def _tokenize(self, text: str) -> list[str]:
        return re.findall(r'\w+', text.lower())

    def _build_index(self):
        """重建 TF-IDF 索引。"""
        N = len(self._chunks)
        if N == 0:
            return
        # DF
        df: dict[str, int] = defaultdict(int)
        chunk_tokens = []
        for chunk in self._chunks:
            toks = self._tokenize(chunk["text"])
            chunk_tokens.append(toks)
            for t in set(toks):
                df[t] += 1
        self._vocab = {t: i for i, t in enumerate(df.keys())}
        self._idf = {t: math.log((N + 1) / (cnt + 1)) + 1 for t, cnt in df.items()}
        # TF-IDF vectors
        self._tfidf_matrix = []
        for toks in chunk_tokens:
            tf: dict[str, float] = defaultdict(float)
            for t in toks:
                tf[t] += 1
            total = len(toks) or 1
            vec = {t: (cnt / total) * self._idf.get(t, 1) for t, cnt in tf.items()}
            self._tfidf_matrix.append(vec)

    def _cosine_scores(self, query: str) -> list[tuple[int, float]]:
        q_toks = self._tokenize(query)
        q_tf: dict[str, float] = defaultdict(float)
        for t in q_toks:
            q_tf[t] += 1
        total = len(q_toks) or 1
        q_vec = {t: (cnt / total) * self._idf.get(t, 1) for t, cnt in q_tf.items()}
        q_norm = math.sqrt(sum(v * v for v in q_vec.values())) or 1
        results = []
        for idx, doc_vec in enumerate(self._tfidf_matrix):
            dot = sum(q_vec.get(t, 0) * v for t, v in doc_vec.items())
            d_norm = math.sqrt(sum(v * v for v in doc_vec.values())) or 1
            results.append((idx, dot / (q_norm * d_norm)))
        return results

    def _index_file(self) -> Path:
        return self.kb_path / f"{self.namespace}_index.json"

    def _save_index(self):
        with open(self._index_file(), "w", encoding="utf-8") as f:
            json.dump(self._chunks, f, ensure_ascii=False)

    def _load_index(self):
        fp = self._index_file()
        if fp.exists():
            with open(fp, encoding="utf-8") as f:
                self._chunks = json.load(f)
            if self._chunks:
                self._build_index()


# ── 验证 ──
rag_tool = RAGTool(
    knowledge_base_path="./tutorial_rag_kb",
    rag_namespace="chapter8_tutorial"
)
print("✅ RAGTool 初始化完成")
print(f"   知识库路径: ./tutorial_rag_kb")
print(f"   命名空间: chapter8_tutorial")
print()
print(rag_tool.run({"action": "status"}))


✅ RAGTool 初始化完成
   知识库路径: ./tutorial_rag_kb
   命名空间: chapter8_tutorial

📊 RAGTool 状态 (namespace=chapter8_tutorial)
  总块数: 0
  词汇量: 0
  知识库路径: tutorial_rag_kb
  已索引文件：


In [11]:
import tempfile, os

rag_tool = RAGTool(
    knowledge_base_path="./tutorial_rag_kb",
    rag_namespace="chapter8_tutorial"
)

# 创建示例文档（模拟真实知识库文档）
tmp_dir = tempfile.mkdtemp()

doc1_path = os.path.join(tmp_dir, "python_guide.md")
with open(doc1_path, "w", encoding="utf-8") as f:
    f.write("""# Python 编程指南

## 基础语法
Python 是一种解释型、高级编程语言，以简洁清晰著称。

## 变量和数据类型
- 整数：`42`
- 字符串：`"Hello World"`
- 列表：`[1, 2, 3]`
- 字典：`{"key": "value"}`

## 函数定义
用 def 关键字定义函数，示例：
def greet(name):
    return "Hello, " + name

## 面向对象编程
Python 支持面向对象编程，通过 class 定义类：
class Person:
    def __init__(self, name):
        self.name = name
    def greet(self):
        return "我叫" + self.name
""")

doc2_path = os.path.join(tmp_dir, "ai_concepts.txt")
with open(doc2_path, "w", encoding="utf-8") as f:
    f.write("""AI 核心概念

机器学习是让计算机从数据中自动学习规律的技术。

深度学习是机器学习的子领域，使用多层神经网络处理复杂数据。
典型架构包括：CNN（图像）、RNN（序列）、Transformer（语言）。

大语言模型（LLM）基于 Transformer 架构，通过海量文本训练，
能够理解和生成自然语言，代表性模型有 GPT、Claude、Qwen 等。

AI Agent 是能够感知环境、制定计划并执行动作的智能体，
通常由 LLM + 工具 + 记忆 + 规划能力构成。
""")

# 将文档添加进知识库
print("📚 向知识库添加文档...")

r1 = rag_tool.run({
    "action": "add_document",
    "file_path": doc1_path,
    "metadata": {"category": "programming", "language": "python"}
})
print(f"文档1 (Python指南): {r1}")

r2 = rag_tool.run({
    "action": "add_document",
    "file_path": doc2_path,
    "metadata": {"category": "ai", "topic": "concepts"}
})
print(f"文档2 (AI概念): {r2}")

print("\n📊 添加后状态:")
print(rag_tool.run({"action": "status"}))


📚 向知识库添加文档...
文档1 (Python指南): ✅ 已添加 5 个分块 (来源: python_guide.md，总块数: 5)
文档2 (AI概念): ✅ 已添加 5 个分块 (来源: ai_concepts.txt，总块数: 10)

📊 添加后状态:
📊 RAGTool 状态 (namespace=chapter8_tutorial)
  总块数: 10
  词汇量: 66
  知识库路径: tutorial_rag_kb
  已索引文件：
    · python_guide.md  (5 块)
    · ai_concepts.txt  (5 块)


In [12]:
rag_tool = RAGTool(
    knowledge_base_path="./tutorial_rag_kb",
    rag_namespace="chapter8_tutorial"
)

# 纯检索（不用 LLM 生成，只返回最相关的文本块）
print("🔍 知识库检索（不调用 LLM）")
print("-" * 40)

queries = ["Python 函数怎么定义", "什么是深度学习", "AI Agent 是什么"]
for q in queries:
    result = rag_tool.run({
        "action": "search",
        "query": q,
        "top_k": 2
    })
    print(f"\n问题: {q}")
    print(result[:300] + "..." if len(result) > 300 else result)


🔍 知识库检索（不调用 LLM）
----------------------------------------

问题: Python 函数怎么定义
[1] 来源: python_guide.md  相关度: 0.534
    # Python 编程指南

[2] 来源: python_guide.md  相关度: 0.312
    ## 基础语法
Python 是一种解释型、高级编程语言，以简洁清晰著称。

问题: 什么是深度学习
[1] 来源: python_guide.md  相关度: 0.000
    # Python 编程指南

[2] 来源: python_guide.md  相关度: 0.000
    ## 基础语法
Python 是一种解释型、高级编程语言，以简洁清晰著称。

问题: AI Agent 是什么
[1] 来源: ai_concepts.txt  相关度: 0.435
    AI Agent 是能够感知环境、制定计划并执行动作的智能体，
通常由 LLM + 工具 + 记忆 + 规划能力构成。

[2] 来源: ai_concepts.txt  相关度: 0.404
    AI 核心概念


## 6. RAGTool：智能问答

`query` 操作 = 检索 + LLM 生成，完整的 RAG 流程：

```
用户问题
  → 向量相似度检索相关文本块
     → 组装 Prompt：「根据以下资料回答问题...」
        → LLM 生成答案
           → 返回给用户
```

> ⚠️ 此操作需要配置 API 密钥

In [13]:
rag_tool = RAGTool(
    knowledge_base_path="./tutorial_rag_kb",
    rag_namespace="chapter8_tutorial"
)

# 完整 RAG 问答（检索 + LLM 生成）
print("🤖 RAG 智能问答")
print("=" * 40)

questions = [
    "Python 中怎么定义一个类？",
    "大语言模型和深度学习有什么关系？",
]

for question in questions:
    print(f"\n❓ 问题: {question}")
    result = rag_tool.run({
        "action": "query",
        "query": question,
        "top_k": 3
    })
    print(f"💬 回答: {result}")


🤖 RAG 智能问答

❓ 问题: Python 中怎么定义一个类？
💬 回答: 根据参考资料 [3]，Python 支持面向对象编程，通过 class 定义类。例如：

```python
class Person:
    def __init__(self, name):
        self.name = name
    def greet(self):
        return "我叫" + self.name
```

在上述代码中，通过 class 关键字定义了一个名为 Person 的类。

❓ 问题: 大语言模型和深度学习有什么关系？
💬 回答: 参考资料中没有关于“大语言模型和深度学习关系”的相关信息。


## 7. Agent 集成 MemoryTool + RAGTool

将 MemoryTool 和 RAGTool 一起注册进 Agent，让 Agent 具备：
- **记忆能力**：记住对话历史、用户偏好
- **知识检索能力**：从文档库检索专业知识回答问题

```
用户输入
  → Agent 判断：需要记忆？需要检索知识库？
     → 调用 MemoryTool 或 RAGTool
        → 整合结果生成回答
           → 将对话存入记忆
```

In [14]:
from hello_agents import HelloAgentsLLM, ToolRegistry

# 1. 初始化工具（使用已实现的 MemoryTool / RAGTool 类）
print("🔧 初始化工具...")
memory_tool = MemoryTool(
    user_id="agent_user",
    memory_types=["working", "episodic", "semantic", "perceptual"]
)
rag_tool = RAGTool(
    knowledge_base_path="./tutorial_rag_kb",
    rag_namespace="chapter8_tutorial"
)
print("✅ MemoryTool 和 RAGTool 已初始化")

# 2. 注册到 ToolRegistry
print("\n🔌 注册工具演示（适配 hello-agents ToolRegistry）...")
registry = ToolRegistry()
# ToolRegistry 使用 register_tool(tool)，tool 要有 .name 属性
# memory_tool / rag_tool 是我们自己的类，给它们加一个 name 属性适配
memory_tool.name = "memory"
rag_tool.name    = "rag"
registry.register_tool(memory_tool)
registry.register_tool(rag_tool)
print(f"✅ 已注册工具: {registry.list_tools()}")

# 3. 演示如何在 Agent 的 system_prompt 中说明工具用途
print("\n🤖 工具信息")
print(f"  MemoryTool 支持的操作: add / search / consolidate / forget / stats")
print(f"  RAGTool    支持的操作: add_document / search / query / status")


🔧 初始化工具...
✅ MemoryTool 和 RAGTool 已初始化

🔌 注册工具演示（适配 hello-agents ToolRegistry）...
✅ 工具 'memory' 已注册。
✅ 工具 'rag' 已注册。
✅ 已注册工具: ['memory', 'rag']

🤖 工具信息
  MemoryTool 支持的操作: add / search / consolidate / forget / stats
  RAGTool    支持的操作: add_document / search / query / status


In [15]:
# 直接用工具演示：记忆存储 + 检索问答组合

memory_tool = MemoryTool(
    user_id="combined_demo_user",
    memory_types=["working", "episodic", "semantic"]
)
rag_tool = RAGTool(
    knowledge_base_path="./tutorial_rag_kb",
    rag_namespace="chapter8_tutorial"
)

# 模拟用户与 Agent 的对话流程
print("=" * 50)
print("模拟对话流程")
print("=" * 50)

# 步骤 1：用户提问，Agent 先检索知识库
user_input = "Python 面向对象编程怎么写？"
print(f"\n👤 用户: {user_input}")

rag_result = rag_tool.run({"action": "search", "query": user_input, "top_k": 2})
print(f"\n📚 检索到相关知识:\n{rag_result[:300]}...")

# 步骤 2：将本次对话存入记忆
memory_tool.run({
    "action": "add",
    "content": f"用户询问了：{user_input}",
    "memory_type": "episodic",
    "importance": 0.7,
    "event_type": "conversation"
})
memory_tool.run({
    "action": "add",
    "content": "用户对 Python OOP 感兴趣",
    "memory_type": "semantic",
    "importance": 0.8,
    "concept": "user_preference",
    "domain": "programming"
})
print("\n🧠 已将对话信息存入记忆")

# 步骤 3：第二轮对话，利用记忆做个性化服务
print("\n" + "-" * 50)
print("👤 用户: 还有什么相关话题推荐吗？")

memory_results = memory_tool.run({
    "action": "search",
    "query": "用户偏好",
    "memory_type": "semantic",
    "limit": 3
})
print(f"\n🧠 从记忆中检索到用户偏好:\n{memory_results}")

print("\n📊 当前记忆状态:")
print(memory_tool.run({"action": "stats"}))


模拟对话流程

👤 用户: Python 面向对象编程怎么写？

📚 检索到相关知识:
[1] 来源: python_guide.md  相关度: 0.534
    # Python 编程指南

[2] 来源: python_guide.md  相关度: 0.312
    ## 基础语法
Python 是一种解释型、高级编程语言，以简洁清晰著称。...

🧠 已将对话信息存入记忆

--------------------------------------------------
👤 用户: 还有什么相关话题推荐吗？

🧠 从记忆中检索到用户偏好:
[semantic] 用户对 Python OOP 感兴趣（重要性=0.8, 相关度=0.160）

📊 当前记忆状态:
📊 记忆统计 (user=combined_demo_user)  总计: 2 条
  working     :   0 条  平均重要性=0.00
  episodic    :   1 条  平均重要性=0.70
  semantic    :   1 条  平均重要性=0.80


In [16]:
# === 工作记忆容量管理演示 ===
# WorkingMemory 默认最多存 50 条，超过会自动按重要性淘汰低优先级记忆

wm_tool = MemoryTool(
    user_id="capacity_demo",
    memory_types=["working"]
)

print("💭 工作记忆容量管理演示")
print("-" * 40)
print("特点：")
print("• 访问速度极快（纯内存）")
print("• 容量上限 50 条")
print("• TTL 自动过期（默认 60 分钟）")
print("• 重要性排序，低优先级先淘汰")

print("\n📝 批量写入测试数据...")
for i in range(10):
    importance = round(0.3 + i * 0.07, 2)
    wm_tool.run({
        "action": "add",
        "content": f"工作任务 {i+1}：优先级 {importance}",
        "memory_type": "working",
        "importance": importance,
        "task_id": i + 1
    })

print("\n📊 当前状态:")
print(wm_tool.run({"action": "stats"}))

print("\n🔍 检索测试（按相关度+重要性排序）:")
result = wm_tool.run({
    "action": "search",
    "query": "工作任务",
    "memory_type": "working",
    "limit": 5
})
print(result)


💭 工作记忆容量管理演示
----------------------------------------
特点：
• 访问速度极快（纯内存）
• 容量上限 50 条
• TTL 自动过期（默认 60 分钟）
• 重要性排序，低优先级先淘汰

📝 批量写入测试数据...

📊 当前状态:
📊 记忆统计 (user=capacity_demo)  总计: 10 条
  working     :  10 条  平均重要性=0.61

🔍 检索测试（按相关度+重要性排序）:
[working] 工作任务 10：优先级 0.93（重要性=0.93, 相关度=0.386）
[working] 工作任务 9：优先级 0.86（重要性=0.86, 相关度=0.372）
[working] 工作任务 8：优先级 0.79（重要性=0.79, 相关度=0.358）
[working] 工作任务 7：优先级 0.72（重要性=0.72, 相关度=0.344）
[working] 工作任务 6：优先级 0.65（重要性=0.65, 相关度=0.330）


---

## 🎉 本章小结

| 工具 | 核心操作 | 使用场景 |
|------|----------|----------|
| **MemoryTool** | `add` / `search` / `consolidate` / `forget` | 对话记忆、用户偏好、知识积累 |
| **RAGTool** | `add_document` / `search` / `query` | 文档问答、知识库检索 |

### 四种记忆类型速查

| 类型 | 关键字段 | 适用场景 |
|------|----------|----------|
| `working` | `importance`、TTL | 当前对话临时上下文 |
| `episodic` | `event_type`、`location` | 历史对话、事件记录 |
| `semantic` | `concept`、`domain` | 长期知识、用户偏好 |
| `perceptual` | `modality`、`source` | 处理过的文件记录 |

### 整合流程
```
工作记忆（短期）→ consolidate → 情景记忆（长期）
                      ↑
              重要性阈值筛选
```

> **下一步**：回顾第八章代码目录 `code/chapter8/`，里面有 11 个示例脚本，可以运行探索更多细节。